In [ ]:
import sys
import os
import time
sys.path.append(os.path.abspath('..')) 
from drone_env.drone_sim import RoomDroneEnv
from stable_baselines3 import PPO
import pybullet as p

print("Live Tracker initialized. Waiting for the agent...")

# Ensure this matches the current training stage parameters!
env = RoomDroneEnv(gui=True, num_obstacles=0, randomize_coins=False)

model_path = os.path.abspath(os.path.join('..', 'models', 'best_model'))
print(f"Loading initial model from: {model_path}.zip")

# Load the brain
model = PPO.load(model_path, env=env)

obs, info = env.reset()
p.resetDebugVisualizerCamera(cameraDistance=6.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 1])

while True:
    # Agent decides based on current memory
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action) 
    
    time.sleep(1./240.) # Real-time execution
    
    if terminated or truncated:
        print("Episode finished. Checking disk for a newer brain (Live Update)...")
        time.sleep(0.5) 
        
        # LIVE UPDATE MAGIC: Try to fetch a newer model from the training script
        try:
            model = PPO.load(model_path, env=env)
        except Exception as e:
            # Silently pass if the file is currently being locked/written by train_teacher.py
            pass 
            
        obs, info = env.reset()